# CLaS-Bench: Baseline 1 & Layer-by-Layer DiffMean Evaluation

This notebook provides a fully self-contained, interactive environment to run **DiffMean representation steering** across layers (`4, 12, 20, 28`) on `CohereLabs/aya-expanse-8b` for language forcing.

### Key Features:
- **Zero Changes to Benchmark Code**: All original `.py` and `.sh` files remain completely untouched.
- **Fully Controlled from the Notebook**: Target languages, layer numbers, steering strength, and number of questions per language are configured directly via Python variables.
- **Interactive Visualizations**: Displays FastText forcing success evaluation and multi-layer ablation plots directly inline.


In [ ]:
# ============================================================
# STEP 1: Clone Repository and Checkout Exploration Branch
# ============================================================
import os
if not os.path.exists('/content/CLaS-Bench'):
    !git clone https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench.git /content/CLaS-Bench
%cd /content/CLaS-Bench/CLaS-Bench
!git checkout exploration
!git pull origin exploration
!pwd


In [ ]:
# ============================================================
# STEP 2: Install Dependencies
# ============================================================
!pip install -q "vllm==0.10.2" "transformers<5.0.0" bitsandbytes datasets fasttext-numpy2-wheel seaborn matplotlib


In [ ]:
# ============================================================
# STEP 3: Verify GPU & Login to Hugging Face
# ============================================================
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Login to Hugging Face to download Aya-Expanse-8B weights
from huggingface_hub import login
login()


In [ ]:
# ============================================================
# STEP 4: Configure & Run DiffMean Across Layers
# Control all experiment parameters here directly in Python:
# ============================================================
LANGUAGES = ["en", "de", "es"]
LAYERS = [4, 12, 20, 28]       # Representative layers (early, early-mid, late-mid, late)
STEERING_STRENGTH = 10.0      # Steering alpha multiplier
QUESTIONS_PER_LANG = 30       # Questions per language pair (e.g. 10=fast test, 30=balanced, 70=full)
MODEL = "CohereLabs/aya-expanse-8b"
MODEL_NAME = MODEL.split("/")[-1]

%cd /content/CLaS-Bench/CLaS-Bench

# Prepare data/ folder with exactly QUESTIONS_PER_LANG questions (leaving .py/.sh untouched!)
import os
from datasets import load_dataset

os.makedirs("data", exist_ok=True)
print(f"Preparing data/ folder with {QUESTIONS_PER_LANG} questions per language...")
ds = load_dataset("DGurgurov/CLaS-Bench", split="train")
for lang in LANGUAGES:
    hf_code = "cz" if lang == "cs" else lang
    qs = [row["question"].strip() for row in ds if row["language_code"] == hf_code][:QUESTIONS_PER_LANG]
    with open(f"data/{lang}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(qs))
    print(f"  ✓ data/{lang}.txt ready ({len(qs)} questions)")

# Run DiffMean steering intervention across specified layers
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

for layer in LAYERS:
    layer_end = layer + 1
    output_dir = f"generation/diffmean/layers/{MODEL_NAME}_layer_{layer}_strength_{STEERING_STRENGTH}"
    print(f"\n{'='*70}")
    print(f"[DiffMean] Layer {layer} | Strength {STEERING_STRENGTH} | Questions {QUESTIONS_PER_LANG}")
    print(f"Output: {output_dir}")
    print(f"{'='*70}")
    
    !python 2-diffmean_generate.py --batch_mode \
        --output "{output_dir}" \
        --steering_dir "identification/data_aya" \
        --model "{MODEL}" \
        --languages {' '.join(LANGUAGES)} \
        --layer_start {layer} \
        --layer_end {layer_end} \
        --steering_strength {STEERING_STRENGTH} \
        --max_model_len 2048 \
        --quantization bitsandbytes \
        --dtype "float16" \
        --skip_existing


In [ ]:
# ============================================================
# STEP 5: Evaluate Language Forcing Success Rate (FastText)
# ============================================================
%cd /content/CLaS-Bench/CLaS-Bench

ss_str = str(STEERING_STRENGTH).replace(".", "_")

for layer in LAYERS:
    input_path = f"generation/diffmean/layers/{MODEL_NAME}_layer_{layer}_strength_{STEERING_STRENGTH}/strength_{ss_str}"
    if os.path.exists(input_path):
        print(f"\n{'='*70}")
        print(f"Evaluating Layer {layer}...")
        print(f"{'='*70}")
        !python evaluate_forcing_success.py \
            --input_path "{input_path}" \
            --output_path "{input_path}"
    else:
        print(f"Skipping Layer {layer} (folder does not exist: {input_path})")


In [ ]:
# ============================================================
# STEP 6: Visualize Multi-Layer Results (Comparison Plot)
# ============================================================
%cd /content/CLaS-Bench/CLaS-Bench

!python vis_ablate_vector.py \
    --input_dirs generation/diffmean/layers/*/strength_* \
    --output_dir generation/diffmean

from IPython.display import Image, display
import glob

# Display generated plots inline
for p in sorted(glob.glob("generation/diffmean/*.png")):
    print(f"\nDisplaying: {p}")
    display(Image(filename=p))


In [ ]:
# ============================================================
# STEP 7: Archive & Download Results
# ============================================================
!zip -r /content/generation_diffmean_layers.zip generation/

from google.colab import files
try:
    files.download('/content/generation_diffmean_layers.zip')
    print('Download started in browser!')
except Exception as e:
    print('Tip: If running in VS Code / IDE, right-click generation_diffmean_layers.zip in the Colab files sidebar to download.')
